In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
import pandas as pd

from src.sentiment_analysis import (
    FinBERTSentiment
)

In [3]:
df = pd.read_csv(
    "/Users/zhengbinheng/Desktop/topic-specific-sentiment/data/processed/topic_assignments.csv"
)

print(df.shape)

df.head()

(32770, 8)


,Headlines,Time,Description,text,clean_text,length,topic,topic_name
0,TikTok considers London and other locations fo...,2020-07-18,TikTok has been in discussions with the UK gov...,TikTok considers London and other locations fo...,tiktok considers london and other locations fo...,47,9,Mergers & Acquisitions
1,Disney cuts ad spending on Facebook amid growi...,2020-07-18,Walt Disney has become the latest company to ...,Disney cuts ad spending on Facebook amid growi...,disney cuts ad spending on facebook amid growi...,56,6,Big Tech & Digital Platforms
2,Trail of missing Wirecard executive leads to B...,2020-07-18,Former Wirecard chief operating officer Jan M...,Trail of missing Wirecard executive leads to B...,trail of missing wirecard executive leads to b...,40,4,Automotive Industry & Corporate Leadership
3,Twitter says attackers downloaded data from up...,2020-07-18,Twitter Inc said on Saturday that hackers were...,Twitter says attackers downloaded data from up...,twitter says attackers downloaded data from up...,47,6,Big Tech & Digital Platforms
4,U.S. Republicans seek liability protections as...,2020-07-17,A battle in the U.S. Congress over a new coron...,U.S. Republicans seek liability protections as...,u s republicans seek liability protections as ...,50,3,US-China Trade War & Geopolitics


In [4]:
df["text"] = (
    df["text"]
    .fillna("")
    .astype(str)
)

In [5]:
model = FinBERTSentiment()

Using device: cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Sanity Check

In [6]:
sample = df.head(10).copy()

sample["sentiment"] = model.predict_batch(
    sample["text"].tolist()
)

sample[
    [
        "Headlines",
        "topic_name",
        "sentiment"
    ]
]

,Headlines,topic_name,sentiment
0,TikTok considers London and other locations fo...,Mergers & Acquisitions,positive
1,Disney cuts ad spending on Facebook amid growi...,Big Tech & Digital Platforms,negative
2,Trail of missing Wirecard executive leads to B...,Automotive Industry & Corporate Leadership,negative
3,Twitter says attackers downloaded data from up...,Big Tech & Digital Platforms,neutral
4,U.S. Republicans seek liability protections as...,US-China Trade War & Geopolitics,positive
5,Wall Street Week Ahead: Fund managers navigate...,Banking & Financial Services,negative
6,Take Five: Hoping for that V-shape in earnings,Corporate Earnings & Equity Markets,negative
7,Evictions nearly back to pre-pandemic levels i...,"COVID-19, Manufacturing & Labor Markets",negative
8,Google bans ads on coronavirus conspiracy theo...,Big Tech & Digital Platforms,negative
9,"Flight to suburbs boosts U.S. homebuilding, bu...","COVID-19, Manufacturing & Labor Markets",positive


Run FinBERT on Full Dataset

In [7]:
df["sentiment"] = model.predict_batch(
    df["text"].tolist(),
    batch_size=32
)

Sentiment Score

In [8]:
df["sentiment_score"] = (
    df["sentiment"]
    .map(model.SCORE_MAPPING)
)

In [9]:
import importlib

import src.sentiment_analysis

importlib.reload(
    src.sentiment_analysis
)

from src.sentiment_analysis import (
    FinBERTSentiment
)

model = FinBERTSentiment()

Using device: cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Probabilities

In [10]:
probs = model.predict_batch_proba(
    df["text"].tolist(),
    batch_size=32
)

In [11]:
prob_df = pd.DataFrame(
    probs,
    columns=[
        "positive_prob",
        "negative_prob",
        "neutral_prob"
    ]
)

In [12]:
df = pd.concat(
    [df, prob_df],
    axis=1
)

Inspect Results

In [13]:
df[
    [
        "Headlines",
        "topic_name",
        "sentiment",
        "sentiment_score",
        "positive_prob",
        "negative_prob",
        "neutral_prob"
    ]
].head()

,Headlines,topic_name,sentiment,sentiment_score,positive_prob,negative_prob,neutral_prob
0,TikTok considers London and other locations fo...,Mergers & Acquisitions,positive,1,0.514773,0.014931,0.470296
1,Disney cuts ad spending on Facebook amid growi...,Big Tech & Digital Platforms,negative,-1,0.007725,0.966416,0.025859
2,Trail of missing Wirecard executive leads to B...,Automotive Industry & Corporate Leadership,negative,-1,0.011173,0.898517,0.090310
3,Twitter says attackers downloaded data from up...,Big Tech & Digital Platforms,neutral,0,0.041731,0.079068,0.879201
4,U.S. Republicans seek liability protections as...,US-China Trade War & Geopolitics,positive,1,0.439223,0.378891,0.181886


Overall Distribution

In [14]:
df["sentiment"].value_counts()

sentiment
negative    15905
positive    10329
neutral      6536
Name: count, dtype: int64

In [15]:
(
    df["sentiment"]
    .value_counts(normalize=True)
    * 100
).round(2)

sentiment
negative    48.54
positive    31.52
neutral     19.95
Name: proportion, dtype: float64

Average Sentiment by Topic

In [16]:
avg_sentiment = (
    df.groupby("topic_name")
      ["sentiment_score"]
      .mean()
      .sort_values()
)

avg_sentiment

topic_name
Financial Regulation & Corporate Scandals    -0.581718
COVID-19, Manufacturing & Labor Markets      -0.424642
Aviation & Air Transportation                -0.244326
US-China Trade War & Geopolitics             -0.224162
Banking & Financial Services                 -0.203369
Macroeconomy & Financial Markets             -0.194210
Automotive Industry & Corporate Leadership   -0.183259
Oil & Energy Markets                         -0.163686
Corporate Earnings & Equity Markets          -0.095839
Big Tech & Digital Platforms                 -0.042610
International Trade & Economic Policy         0.006703
Mergers & Acquisitions                        0.219760
Name: sentiment_score, dtype: float64

Average Negative Probability

In [17]:
negative_scores = (
    df.groupby("topic_name")
      ["negative_prob"]
      .mean()
      .sort_values(
          ascending=False
      )
)

negative_scores

topic_name
Financial Regulation & Corporate Scandals     0.632747
COVID-19, Manufacturing & Labor Markets       0.606224
Macroeconomy & Financial Markets              0.554147
Corporate Earnings & Equity Markets           0.504105
Aviation & Air Transportation                 0.503582
Oil & Energy Markets                          0.480188
Banking & Financial Services                  0.471051
US-China Trade War & Geopolitics              0.463235
Automotive Industry & Corporate Leadership    0.401436
International Trade & Economic Policy         0.381899
Big Tech & Digital Platforms                  0.364469
Mergers & Acquisitions                        0.250240
Name: negative_prob, dtype: float64

Topic × Sentiment

In [18]:
topic_sentiment = pd.crosstab(
    df["topic_name"],
    df["sentiment"]
)

topic_sentiment

sentiment,negative,neutral,positive
topic_name,,,
Automotive Industry & Corporate Leadership,1413,1132,800
Aviation & Air Transportation,1183,430,634
Banking & Financial Services,1225,550,718
Big Tech & Digital Platforms,1287,925,1144
"COVID-19, Manufacturing & Labor Markets",1703,375,576
Corporate Earnings & Equity Markets,1221,165,993
Financial Regulation & Corporate Scandals,1917,492,326
International Trade & Economic Policy,954,463,970
Macroeconomy & Financial Markets,2205,251,1447


In [19]:
topic_sentiment_pct = (
    pd.crosstab(
        df["topic_name"],
        df["sentiment"],
        normalize="index"
    ) * 100
)

topic_sentiment_pct.round(2)

sentiment,negative,neutral,positive
topic_name,,,
Automotive Industry & Corporate Leadership,42.24,33.84,23.92
Aviation & Air Transportation,52.65,19.14,28.22
Banking & Financial Services,49.14,22.06,28.80
Big Tech & Digital Platforms,38.35,27.56,34.09
"COVID-19, Manufacturing & Labor Markets",64.17,14.13,21.70
Corporate Earnings & Equity Markets,51.32,6.94,41.74
Financial Regulation & Corporate Scandals,70.09,17.99,11.92
International Trade & Economic Policy,39.97,19.40,40.64
Macroeconomy & Financial Markets,56.50,6.43,37.07


In [20]:
df.to_csv(
    "/Users/zhengbinheng/Desktop/topic-specific-sentiment/data/processed/sentiment_scores.csv",
    index=False
)

In [21]:
avg_sentiment.to_csv(
    "/Users/zhengbinheng/Desktop/topic-specific-sentiment/results/tables/topic_sentiment.csv"
)